FASE 1

In [2]:
import requests

In [3]:
import pandas as pd

In [4]:
#Defino la URL del endpoint de la API

url= "https://beta.adalab.es/resources/apis/pelis/pelis.json"
#Realizo la petición GET a la API
response= requests.get(url)

In [5]:
if response.status_code == 200:
        datos_peliculas = response.json()

In [6]:
peliculas_100 = datos_peliculas[:100]

In [7]:
df_peliculas = pd.DataFrame(peliculas_100)

In [8]:
df_peliculas.columns


Index(['id', 'titulo', 'año', 'duracion', 'genero', 'adultos', 'subtitulos'], dtype='str')

In [9]:
pip install sqlalchemy pymysql pandas requests


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [10]:
from sqlalchemy import create_engine

In [11]:

usuario = "Checha"
contrasena = "Chechita26"
host = "localhost"
puerto = "3306"
base_datos = "peliculas_adalab"

# Creamos el motor de conexión
conexion_url = f"mysql+pymysql://{usuario}:{contrasena}@{host}:{puerto}/{base_datos}"
engine = create_engine(conexion_url)




In [12]:
# INSERTAR EN LA BASE DE DATOS
with engine.begin() as conexion:
    columna_genero_api = "genero"  
    generos_unicos = df_peliculas[columna_genero_api].dropna().unique()



In [42]:
df_peliculas.columns = df_peliculas.columns.str.strip()

In [13]:
import pandas as pd
from sqlalchemy import text 

# Limpieza de datos en el DataFrame

df_peliculas["genero"] = df_peliculas["genero"].astype(str).str.strip()
df_peliculas["titulo"] = df_peliculas["titulo"].astype(str).str.strip()

with engine.begin() as conexion:
    # Insertar géneros únicos
    generos_unicos = df_peliculas["genero"].dropna().unique()
    for genero in generos_unicos:
        conexion.execute(
            text("INSERT IGNORE INTO tabla_generos (nombre_genero) VALUES (:genero)"),
            {"genero": str(genero)},
        )

    # Recuperar los IDs generados por MySQL
    resultado_generos = conexion.execute(
        text("SELECT id_genero, nombre_genero FROM tabla_generos")
    ).fetchall()
    
    # Limpieza  claves del diccionario 
    mapeo_generos = {str(nombre).strip(): id_gen for id_gen, nombre in resultado_generos}

    # Mapeo del ID
    df_peliculas["id_genero"] = df_peliculas["genero"].map(mapeo_generos)

    # Query de inserción de películas
    query_pelicula = text("""
        INSERT INTO tabla_peliculas (titulo, año, duracion, adultos, id_genero)
        VALUES (:titulo, :ano, :duracion, :adultos, :id_gen)
    """)
    
    for _, row in df_peliculas.iterrows():
        id_genero_valor = int(row["id_genero"]) if pd.notna(row["id_genero"]) else None
        
        conexion.execute(
            query_pelicula,
            {
                "titulo": str(row["titulo"]),
                "ano": int(row["año"]),
                "duracion": int(row["duracion"]),
                "adultos": "Sí" if str(row["adultos"]).lower() in ["true", "sí", "si", "yes", "1"] else "No",
                "id_gen": id_genero_valor,
            },
        )
    
    # Confirmación absoluta en la base de datos
    conexion.execute(text("COMMIT"))

print("Datos insertados")

Datos insertados
